# files of GAM

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/Mtech Project/codes/gam/GAM/gam

/content/drive/MyDrive/Mtech Project/codes/gam/GAM/gam


In [3]:
!pwd

/content/drive/MyDrive/Mtech Project/codes/gam/GAM/gam


In [4]:
!ls

data	matrix.py  optimizer_helper.py	saving.py    smooth_cross_entropy.py  util.py
gam.py	meters.py  __pycache__		schedule.py  train_with_gam.py	      validate.py


In [62]:
import gam
import optimizer_helper
import smooth_cross_entropy as sce



In [63]:
from train_with_gam import train_epoch
from optimizer_helper import get_optim_and_schedulers

In [ ]:
# !git clone https://github.com/xxgege/GAM.git

# codes of LO

## regularizers

In [7]:
import torch
from functools import reduce
import torch.nn.functional as F
from operator import mul
import copy
import torch
import sys
#sys.path.insert(1, '/nlsasfs/home/metalearning/nckris/Suraj_LSAIML/L2O-CL/DMO/GAM_files/gam')

#from gam_util import enable_running_stats, disable_running_stats
# from torch.distributed import ReduceOp
# import contextlib




def get_flat_params(model):
    params = []
    for name, module in model.named_modules():
        if len(module._parameters) != 0:
            params.append(module._parameters['weight'].view(-1))
            try:
                params.append(module._parameters['bias'].view(-1))
            except:
                pass
    return torch.cat(params)



def set_flat_params(model, flat_params):
    # Restore original shapes
    offset = 0
    for module in model.modules():
        if len(module._parameters) != 0:
            weight_shape = module._parameters['weight'].size()
            weight_flat_size = reduce(mul, weight_shape, 1)
            module._parameters['weight'] = flat_params[
                                            offset:offset + weight_flat_size].view(*weight_shape)
            try:
                bias_shape = module._parameters['bias'].size()
                bias_flat_size = reduce(mul, bias_shape, 1)
                module._parameters['bias'] = flat_params[
                                                offset + weight_flat_size:offset + weight_flat_size + bias_flat_size].view(
                    *bias_shape)
            except:
                bias_flat_size = 0
            offset += weight_flat_size + bias_flat_size
    return model




class SAM(torch.optim.Optimizer):
    def __init__(self, params, base_optimizer, rho=0.05, adaptive=False, **kwargs): #
        assert rho >= 0.0, f"Invalid rho, should be non-negative: {rho}"

        defaults = dict(rho=rho, adaptive=adaptive, **kwargs)
        super(SAM, self).__init__(params, defaults)
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups
        self.defaults.update(self.base_optimizer.defaults)

    @torch.no_grad()
    def first_step(self, zero_grad=False):

        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + 1e-12)

            for p in group["params"]:
                if p.grad is None: continue
                self.state[p]["old_p"] = p.data.clone()
                e_w = (torch.pow(p, 2) if group["adaptive"] else 1.0) * p.grad * scale.to(p)
                p.add_(e_w)  # climb to the local maximum "w + e(w)"

        if zero_grad: self.zero_grad()



    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None: continue
                p.data = self.state[p]["old_p"]  # get back to "w" from "w + e(w)"
                # print(p.grad)
        # self.base_optimizer.step()  # do the actual "sharpness-aware" update
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def step(self, closure=None):
        assert closure is not None, "Sharpness Aware Minimization requires closure, but it was not provided"
        closure = torch.enable_grad()(closure)  # the closure should do a full forward-backward pass

        self.first_step(zero_grad=True)
        closure()
        self.second_step()

    def _grad_norm(self):
        shared_device = self.param_groups[0]["params"][0].device  # put everything on the same device, in case of model parallelism

        norm = torch.norm(
                    torch.stack([
                        ((torch.abs(p) if group["adaptive"] else 1.0) * p.grad).norm(p=2).to(shared_device)
                        for group in self.param_groups for p in group["params"]
                        if p.grad is not None
                    ]),
                    p=2
               )
        return norm

    def load_state_dict(self, state_dict):
        super().load_state_dict(state_dict)  #takes in optimizer.state_dicts
        self.base_optimizer.param_groups = self.param_groups




class smooth_regularize:

    '''
    lamb: regularization coefficient
    eps: perturbation length
    num_pga_steps: number of projected gradient ascent steps
    optimizer_model : the learned optimizer

    '''
    def __init__(self, lamb, eps, num_pga_steps,optimizer_model):
        self.lamb = lamb
        self.eps = eps
        self.num_pga_steps = num_pga_steps
        self.optimizer = optimizer_model

    def run(self, model_with_grads):
        grads = []
        for name, module in model_with_grads.named_modules():
            # print(name)
            if len(module._parameters) != 0:
                grads.append(module._parameters['weight'].grad.data.view(-1).unsqueeze(-1))
                try:
                    grads.append(module._parameters['bias'].grad.data.view(-1).unsqueeze(-1))
                except:
                    pass

        flat_grads = torch.cat(grads)
        second_moment = (flat_grads**2).mean() + 1e-5
        flat_grads = flat_grads / torch.sqrt(second_moment)
        inputs = flat_grads.view(-1, 1).unsqueeze(1)

        state_adv = inputs.detach() + 0.01 * torch.randn(inputs.shape).cuda().detach()
        step_size = self.eps / 5
        # ori_h0 = optimizer_model.h0
        ori_h0 = self.optimizer.h0.data
        ori_c0 = self.optimizer.c0.data    # saving a copy of the hidden states since they get changed after one forward pass
                                            # need to update it again later while finding the distance between two updates

        for _ in range(self.num_pga_steps):
            state_adv.requires_grad_()
            with torch.enable_grad():
                perturbed_update = self.optimizer(state_adv)    #optimizer_model(state_adv)#
                self.optimizer.h0 = ori_h0                      #optimizer_model.h0 = ori_h0 #
                self.optimizer.c0 = ori_c0
                # print(self.optimizer.h0[0])
                update = self.optimizer(inputs)                 #optimizer_model(inputs)#
                self.optimizer.h0 = ori_h0                      #optimizer_model.h0 = ori_h0 #
                self.optimizer.c0 = ori_c0
                # print(self.optimizer.h0[0][1][1])
                distance = F.mse_loss(perturbed_update, update)
                # print(state_adv)
                grad = torch.autograd.grad(distance, [state_adv], allow_unused=True)[0]

                state_adv = state_adv.detach() + step_size * torch.sign(grad.detach())
                state_adv = torch.min(torch.max(state_adv, inputs - self.eps), inputs + self.eps)

        self.optimizer.h0 = ori_h0                              #optimizer_model.h0 = ori_h0 #
        self.optimizer.c0 = ori_c0
        perturbed_inputs = state_adv.detach()
        perturbed_update = self.optimizer(perturbed_inputs)     #optimizer_model(perturbed_inputs)#
        self.optimizer.h0 = ori_h0                              #optimizer_model.h0 = ori_h0 #
        self.optimizer.c0 = ori_c0
        update = self.optimizer(inputs)                         #optimizer_model(inputs) #
        distance = F.mse_loss(update, perturbed_update)


        return  distance





## meta optimizer

In [8]:
from functools import reduce
from operator import mul

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

print(torch.cuda.is_available(), device)


def initialize_rnn_hidden_state(dim_hid,n_layers,n_params):
    '''
    dim_his : hidden state dimension (for each param)
    n_layers : number of layers of LSTM
    n_params : number of parameters in the optimizee model
    '''
    h0 = torch.zeros(n_layers,n_params,dim_hid).requires_grad_(requires_grad=False).to(device)
    c0 = torch.zeros(n_layers,n_params,dim_hid).requires_grad_(requires_grad=False).to(device)

    return h0, c0




class MetaOptimizerLSTM(nn.Module):

    def __init__(self, model, num_layers, hidden_size, preproc_factor = 10.0, preproc =True):
        super(MetaOptimizerLSTM, self).__init__()
        self.meta_model = model
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.preproc = preproc

        if self.preproc:
            self.first_order = nn.LSTM(2,self.hidden_size,self.num_layers,batch_first=True,bias=False)
        else:
            self.first_order = nn.LSTM(1,self.hidden_size,self.num_layers,batch_first=True,bias=False)

        self.outputer = nn.Linear(hidden_size,1,bias=False)
        self.outputer.weight.data.mul_(0.1)
        self.preproc_factor = preproc_factor
        self.preproc_threshold = np.exp(-preproc_factor)

    def reset_lstm(self, keep_states=False, model=None):
        self.meta_model.reset()
        self.meta_model.copy_params_from(model)
        if keep_states :
            self.h0 = self.h0.data ####detaches and renews the computational graph (is doesnt require grad but helps in finding the grad for others)
            self.c0 = self.c0.data
        else:
            self.h0, self.c0 = initialize_rnn_hidden_state(self.hidden_size, self.num_layers, self.meta_model.get_flat_params().size(0))

    def forward(self, x):
        # print('\n', self.preproc_factor, self.preproc_threshold, '\n')
        # print(x.size())
        if self.preproc:
            inp = torch.zeros(x.size()[0], 2).to(device)

            #boolean to see which gradients pass the threshold (gradient cap)
            keep_grads = (torch.abs(x) >= self.preproc_threshold).squeeze()
            # print(inp.size(), keep_grads.size())

            inp[:, 0][keep_grads] = (torch.log(torch.abs(x[keep_grads]) + 1e-8) / self.preproc_factor).squeeze()

            #keeping all the signs of the gradients that have "true" value in keep_grads
            inp[:, 1][keep_grads] = torch.sign(x[keep_grads]).squeeze()
            inp[:, 0][~keep_grads] = -1
            inp[:, 1][~keep_grads] = (float(np.exp(self.preproc_factor)) * x[~keep_grads]).squeeze()
            x = inp.unsqueeze(1)
            x = x.to(device)
            # print(inp.size())


        # print(self.h0.size(), self.c0.size())
        output1, (hn1,cn1) = self.first_order(x, (self.h0, self.c0))
        # print(output1.size(), hn1.size(), self.h0.size())

        self.h0 = hn1
        self.c0 = cn1
        o1 = self.outputer(output1)
        return o1.squeeze()

    def meta_update(self, model_with_grads):
        # First we need to create a flat version of parameters and gradients
        grads = []
        for name, module in model_with_grads.named_modules():
            # print(name)
            if len(module._parameters) != 0:
                grads.append(module._parameters['weight'].grad.data.view(-1).unsqueeze(-1))
                try:
                    grads.append(module._parameters['bias'].grad.data.view(-1).unsqueeze(-1))
                except:
                    pass

        self.meta_model.copy_params_from(model_with_grads)
        flat_params = self.meta_model.get_flat_params()     #we dont use the actual model model_with_grads to find updated params,
                                                            #else that goes inside the computational graph and will result in updating
                                                            #gradient of meta_optimizer in the TBTT
        flat_grads = torch.cat(grads)

        # second_moment = (flat_grads**2).mean() + 1e-5
        # flat_grads = flat_grads / torch.sqrt(second_moment)
        inputs = flat_grads.view(-1, 1).unsqueeze(1) #Variable(flat_grads.view(-1, 1).unsqueeze(1))

        update = self(inputs)


        flat_params = flat_params + update

        self.meta_model.set_flat_params(flat_params)

        # Finally, copy values from the meta model to the normal one.
        self.meta_model.copy_params_to(model_with_grads)
        return self.meta_model.model   #since self.meta_model is MetaModel(meta_model)





# A helper class that keeps track of meta updates
# It's done by replacing parameters with variables and applying updates to
# them.
class MetaModel:

    def __init__(self, model):
        self.model = model

    def parameters(self):
        return self.model.parameters()

    def reset(self):
        for module in self.model.modules():
            if len(module._parameters) != 0:
                module._parameters['weight'] = module._parameters['weight'].data
                try:
                    module._parameters['bias'] = module._parameters['bias'].data
                except:
                    pass


    def get_flat_params(self):
        params = []
        for name, module in self.model.named_modules():
            if len(module._parameters) != 0:
                params.append(module._parameters['weight'].view(-1))
                try:
                    params.append(module._parameters['bias'].view(-1))
                except:
                    pass
        return torch.cat(params)


    def set_flat_params(self, flat_params):
        # Restore original shapes
        offset = 0
        for module in self.model.modules():
            if len(module._parameters) != 0:
                weight_shape = module._parameters['weight'].size()
                weight_flat_size = reduce(mul, weight_shape, 1)
                module._parameters['weight'] = flat_params[
                                               offset:offset + weight_flat_size].view(*weight_shape)
                try:
                    bias_shape = module._parameters['bias'].size()
                    bias_flat_size = reduce(mul, bias_shape, 1)
                    module._parameters['bias'] = flat_params[
                                                 offset + weight_flat_size:offset + weight_flat_size + bias_flat_size].view(
                        *bias_shape)
                except:
                    bias_flat_size = 0
                offset += weight_flat_size + bias_flat_size


    def copy_params_from(self, model):
        for modelA, modelB in zip(self.model.parameters(), model.parameters()):
            modelA.data.copy_(modelB.data)

    def copy_params_to(self, model):
        for modelA, modelB in zip(self.model.parameters(), model.parameters()):
            modelB.data.copy_(modelA.data)


True cuda:0


## Models

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from functools import reduce
from operator import mul
import torch.optim as optim


def initialize_model(dataset='mnist', model_name = 'mlp'):

    assert dataset == 'mnist' or dataset == 'cifar10', 'choose either mnist or cifar10'

    if dataset=='mnist':
        num_classes = 10
    elif dataset == 'cifar10':
        num_classes = 10

    if (model_name == 'mlp'):
        return Model(num_classes, dataset)
    elif (model_name == 'deep_mlp'):
        return ModelDeep(num_classes, dataset)
    elif (model_name == 'conv'):
        return Conv_NN(num_classes, dataset)
    elif (model_name == 'conv_large'):
        return Conv_NN_Large(num_classes, dataset)
    elif (model_name == 'mlp_Relu'):
        return Model_Relu(num_classes, dataset)


class Model(nn.Module):

    def __init__(self, num_classes, dataset):
        super(Model, self).__init__()
        self.dataset = dataset

        if self.dataset == 'mnist':
            self.linear1 = nn.Linear(28 * 28, 100)
        elif self.dataset == 'cifar10':
            self.linear1 = nn.Linear(32*32*3, 100)

        self.linear2 = nn.Linear(100, num_classes)



    def forward(self, inputs):
        if self.dataset == 'mnist':
            x = inputs.view(-1, 28 * 28)
        elif self.dataset == 'cifar10':
            x = inputs.view(-1, 32 * 32 * 3)

        x = F.sigmoid(self.linear1(x))
        x = self.linear2(x)
        return F.log_softmax(x, dim=-1)

    def get_flat_params(self):
        params = []

        for module in self.children():
            params.append(module._parameters['weight'].view(-1))
            params.append(module._parameters['bias'].view(-1))

        return torch.cat(params)

    def set_flat_params(self, flat_params):
        # Restore original shapes
        offset = 0
        for i, module in enumerate(self.children()):
            weight_shape = module._parameters['weight'].size()
            bias_shape = module._parameters['bias'].size()

            weight_flat_size = reduce(mul, weight_shape, 1)
            bias_flat_size = reduce(mul, bias_shape, 1)

            module._parameters['weight'].data = flat_params[
                offset:offset + weight_flat_size].view(*weight_shape).data
            module._parameters['bias'].data = flat_params[
                offset + weight_flat_size:offset + weight_flat_size + bias_flat_size].view(*bias_shape).data

            offset += weight_flat_size + bias_flat_size

class Model_Relu(nn.Module):

    def __init__(self, num_classes, dataset):
        super(Model_Relu, self).__init__()
        self.dataset = dataset

        if self.dataset == 'mnist':
            self.linear1 = nn.Linear(28 * 28, 100)
        elif self.dataset == 'cifar10':
            self.linear1 = nn.Linear(32*32*3, 100)
        self.linear2 = nn.Linear(100, int(num_classes))

    def forward(self, inputs):
        if self.dataset == 'mnist':
            x = inputs.view(-1, 28 * 28)
        elif self.dataset == 'cifar10':
            x = inputs.view(-1, 32 * 32 * 3)
        x = F.relu(self.linear1(x))
        x = self.linear2(x)
        return F.log_softmax(x, dim=-1)

    def get_flat_params(self):
        params = []

        for module in self.children():
            params.append(module._parameters['weight'].view(-1))
            params.append(module._parameters['bias'].view(-1))

        return torch.cat(params)

    def set_flat_params(self, flat_params):
        # Restore original shapes
        offset = 0
        for i, module in enumerate(self.children()):
            weight_shape = module._parameters['weight'].size()
            bias_shape = module._parameters['bias'].size()

            weight_flat_size = reduce(mul, weight_shape, 1)
            bias_flat_size = reduce(mul, bias_shape, 1)

            module._parameters['weight'].data = flat_params[
                offset:offset + weight_flat_size].view(*weight_shape).data
            module._parameters['bias'].data = flat_params[
                offset + weight_flat_size:offset + weight_flat_size + bias_flat_size].view(*bias_shape).data

            offset += weight_flat_size + bias_flat_size


class ModelDeep(nn.Module):

    def __init__(self, num_classes, dataset):
        super(ModelDeep, self).__init__()
        self.dataset = dataset

        if self.dataset == 'mnist':
            self.linear1 = nn.Linear(28 * 28, 32)
        elif self.dataset == 'cifar10':
            self.linear1 = nn.Linear(32*32*3, 32)

        self.linear2 = nn.Linear(32, 32)
        self.linear3 = nn.Linear(32, num_classes)

    def forward(self, inputs):
        if self.dataset == 'mnist':
            x = inputs.view(-1, 28 * 28)
        elif self.dataset == 'cifar10':
            x = inputs.view(-1, 32 * 32 * 3)

        x = F.relu(self.linear1(x))
        x = F.relu(self.linear2(x))
        x = self.linear3(x)
        return F.log_softmax(x, dim=-1)

    def get_flat_params(self):
        params = []

        for module in self.children():
            params.append(module._parameters['weight'].view(-1))
            params.append(module._parameters['bias'].view(-1))

        return torch.cat(params)

    def set_flat_params(self, flat_params):
        # Restore original shapes
        offset = 0
        for i, module in enumerate(self.children()):
            weight_shape = module._parameters['weight'].size()
            bias_shape = module._parameters['bias'].size()

            weight_flat_size = reduce(mul, weight_shape, 1)
            bias_flat_size = reduce(mul, bias_shape, 1)

            module._parameters['weight'].data = flat_params[
                offset:offset + weight_flat_size].view(*weight_shape).data
            module._parameters['bias'].data = flat_params[
                offset + weight_flat_size:offset + weight_flat_size + bias_flat_size].view(*bias_shape).data

            offset += weight_flat_size + bias_flat_size


class Conv_NN(nn.Module):

    def __init__(self, num_classes, data):
        super(Conv_NN, self).__init__()

        # self.conv3 = nn.Conv2d(32, 32, kernel_size=3)
        # self.linear1 = nn.Linear(128, 10)
        if (data == 'mnist'):
            self.conv1 = nn.Conv2d(1, 16, kernel_size=3)
            self.fc1 = nn.Linear(11*11*32, 128)
        elif (data == 'cifar10'):
            self.conv1 = nn.Conv2d(3, 16, kernel_size=3)
            self.fc1 = nn.Linear(13*13*32, 128)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=5)
        self.fc2 = nn.Linear(128, num_classes)


    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        # x = F.relu(self.conv3(x))
        x = F.avg_pool2d(x, 2, 2)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))

        x = self.fc2(x)
        return F.log_softmax(x, dim=-1)

    def get_flat_params(self):
        params = []
        for name, module in self.named_modules():
            if len(module._parameters) != 0:
                params.append(module._parameters['weight'].view(-1))
                try:
                    params.append(module._parameters['bias'].view(-1))
                except:
                    pass

        return torch.cat(params)

    def set_flat_params(self, flat_params):
        # Restore original shapes
        offset = 0
        for module in self.modules():
            if len(module._parameters) != 0:
                weight_shape = module._parameters['weight'].size()
                weight_flat_size = reduce(mul, weight_shape, 1)
                module._parameters['weight'].data = flat_params[
                                               offset:offset + weight_flat_size].view(*weight_shape).data
                try:
                    bias_shape = module._parameters['bias'].size()
                    bias_flat_size = reduce(mul, bias_shape, 1)
                    module._parameters['bias'].data = flat_params[
                                                 offset + weight_flat_size:offset + weight_flat_size + bias_flat_size].view(
                        *bias_shape).data
                except:
                    bias_flat_size = 0
                offset += weight_flat_size + bias_flat_size


class Conv_NN_Large(nn.Module):

    def __init__(self, num_classes, data):
        super(Conv_NN_Large, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3)
        self.conv2 = nn.Conv2d(32, 32, 3)
        self.conv3 = nn.Conv2d(32, 32, 5)
        self.conv4 = nn.Conv2d(32, 32, 5)
        if data == 'mnist':
            self.fc1 = nn.Linear(8*8, 128)
        if data == 'cifar10':
            self.fc1 = nn.Linear(10*10, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        out = F.relu(self.conv1(x))
        out = F.relu(self.conv2(out))
        # out = F.max_pool2d(out, 2, 2)
        out = F.relu(self.conv3(out))
        out = F.relu(self.conv4(out))
        out = F.max_pool2d(out, 2, 2)   #inp, kernel_size, stride_size
        out = out.view(out.size(0), -1)
        out = F.relu(self.fc1(out))
        out = self.fc2(out)
        return F.log_softmax(out, dim=-1)

    def get_flat_params(self):
        params = []
        for name, module in self.named_modules():
            if len(module._parameters) != 0:
                params.append(module._parameters['weight'].view(-1))
                try:
                    params.append(module._parameters['bias'].view(-1))
                except:
                    pass

        return torch.cat(params)

    def set_flat_params(self, flat_params):
        # Restore original shapes
        offset = 0
        for module in self.modules():
            if len(module._parameters) != 0:
                weight_shape = module._parameters['weight'].size()
                weight_flat_size = reduce(mul, weight_shape, 1)
                module._parameters['weight'].data = flat_params[
                                               offset:offset + weight_flat_size].view(*weight_shape).data
                try:
                    bias_shape = module._parameters['bias'].size()
                    bias_flat_size = reduce(mul, bias_shape, 1)
                    module._parameters['bias'].data = flat_params[
                                                 offset + weight_flat_size:offset + weight_flat_size + bias_flat_size].view(
                        *bias_shape).data
                except:
                    bias_flat_size = 0
                offset += weight_flat_size + bias_flat_size

########################################################################################


#dont use
class LeNet(nn.Module):

    def __init__(self):
        super(LeNet, self).__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.conv2_drop = nn.Dropout2d()
        self.fc1 = nn.Linear(320, 50)
        self.fc2 = nn.Linear(50, 10)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2(x), 2))
        x = x.view(-1, 320)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, training=self.training)
        x = self.fc2(x)
        return F.log_softmax(x, dim=-1)

    def get_flat_params(self):
        params = []
        for name, module in self.named_modules():
            if len(module._parameters) != 0:
                params.append(module._parameters['weight'].view(-1))
                try:
                    params.append(module._parameters['bias'].view(-1))
                except:
                    pass

        return torch.cat(params)

    def set_flat_params(self, flat_params):
        # Restore original shapes
        offset = 0
        for module in self.modules():
            if len(module._parameters) != 0:
                weight_shape = module._parameters['weight'].size()
                weight_flat_size = reduce(mul, weight_shape, 1)
                module._parameters['weight'].data = flat_params[
                                               offset:offset + weight_flat_size].view(*weight_shape).data
                try:
                    bias_shape = module._parameters['bias'].size()
                    bias_flat_size = reduce(mul, bias_shape, 1)
                    module._parameters['bias'].data = flat_params[
                                                 offset + weight_flat_size:offset + weight_flat_size + bias_flat_size].view(
                        *bias_shape).data
                except:
                    bias_flat_size = 0
                offset += weight_flat_size + bias_flat_size



class Inception(nn.Module):
    def __init__(self, in_planes, n1x1, n3x3red, n3x3, n5x5red, n5x5, pool_planes):
        super(Inception, self).__init__()
        # 1x1 conv branch
        self.b1 = nn.Sequential(
            nn.Conv2d(in_planes, n1x1, kernel_size=1),
            nn.BatchNorm2d(n1x1),
            nn.ReLU(True),
        )

        # 1x1 conv -> 3x3 conv branch
        self.b2 = nn.Sequential(
            nn.Conv2d(in_planes, n3x3red, kernel_size=1),
            nn.BatchNorm2d(n3x3red),
            nn.ReLU(True),
            nn.Conv2d(n3x3red, n3x3, kernel_size=3, padding=1),
            nn.BatchNorm2d(n3x3),
            nn.ReLU(True),
        )

        # 1x1 conv -> 5x5 conv branch
        self.b3 = nn.Sequential(
            nn.Conv2d(in_planes, n5x5red, kernel_size=1),
            nn.BatchNorm2d(n5x5red),
            nn.ReLU(True),
            nn.Conv2d(n5x5red, n5x5, kernel_size=3, padding=1),
            nn.BatchNorm2d(n5x5),
            nn.ReLU(True),
            nn.Conv2d(n5x5, n5x5, kernel_size=3, padding=1),
            nn.BatchNorm2d(n5x5),
            nn.ReLU(True),
        )

        # 3x3 pool -> 1x1 conv branch
        self.b4 = nn.Sequential(
            nn.MaxPool2d(3, stride=1, padding=1),
            nn.Conv2d(in_planes, pool_planes, kernel_size=1),
            nn.BatchNorm2d(pool_planes),
            nn.ReLU(True),
        )

    def forward(self, x):
        y1 = self.b1(x)
        y2 = self.b2(x)
        y3 = self.b3(x)
        y4 = self.b4(x)
        return torch.cat([y1,y2,y3,y4], 1)


class GoogLeNet(nn.Module):
    def __init__(self):
        super(GoogLeNet, self).__init__()
        self.pre_layers = nn.Sequential(
            nn.Conv2d(3, 192, kernel_size=3, padding=1),
            nn.BatchNorm2d(192),
            nn.ReLU(True),
        )

        self.a3 = Inception(192,  64,  96, 128, 16, 32, 32)
        self.b3 = Inception(256, 128, 128, 192, 32, 96, 64)

        self.maxpool = nn.MaxPool2d(3, stride=2, padding=1)

        self.a4 = Inception(480, 192,  96, 208, 16,  48,  64)
        self.b4 = Inception(512, 160, 112, 224, 24,  64,  64)
        self.c4 = Inception(512, 128, 128, 256, 24,  64,  64)
        self.d4 = Inception(512, 112, 144, 288, 32,  64,  64)
        self.e4 = Inception(528, 256, 160, 320, 32, 128, 128)

        self.a5 = Inception(832, 256, 160, 320, 32, 128, 128)
        self.b5 = Inception(832, 384, 192, 384, 48, 128, 128)

        self.avgpool = nn.AvgPool2d(8, stride=1)
        self.linear = nn.Linear(1024, 10)

    def forward(self, x):
        out = self.pre_layers(x)
        out = self.a3(out)
        out = self.b3(out)
        out = self.maxpool(out)
        out = self.a4(out)
        out = self.b4(out)
        out = self.c4(out)
        out = self.d4(out)
        out = self.e4(out)
        out = self.maxpool(out)
        out = self.a5(out)
        out = self.b5(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return F.log_softmax(out, dim=-1)


cfg = {
    'VGG11': [64, 'M', 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M'],
    'VGG13': [64, 64, 'M', 128, 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M'],
    'VGG16': [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M', 512, 512, 512, 'M', 512, 512, 512, 'M'],
    'VGG19': [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 256, 'M', 512, 512, 512, 512, 'M', 512, 512, 512, 512, 'M'],
}


class VGG(nn.Module):
    def __init__(self, vgg_name):
        super(VGG, self).__init__()
        self.features = self._make_layers(cfg[vgg_name])
        self.classifier = nn.Linear(512, 10)

    def forward(self, x):
        out = self.features(x)
        out = out.view(out.size(0), -1)
        out = self.classifier(out)
        return F.log_softmax(out, dim=-1)

    def _make_layers(self, cfg):
        layers = []
        in_channels = 3
        for x in cfg:
            if x == 'M':
                layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
            else:
                layers += [nn.Conv2d(in_channels, x, kernel_size=3, padding=1),
                           nn.BatchNorm2d(x),
                           nn.ReLU(inplace=True)]
                in_channels = x
        layers += [nn.AvgPool2d(kernel_size=1, stride=1)]
        return nn.Sequential(*layers)



## data loaders

In [15]:
from torchvision import datasets, transforms
import torch
import numpy as np


folder_name = "/content/drive/MyDrive/Mtech Project/codes/gam/GAM/gam/"

def data_loader(data = 'mnist', batch_size = 128, random_seed = 10):

    if data == 'mnist':

        transform_test = transforms.Compose([
            transforms.ToTensor(),
        ])

        trainset = datasets.MNIST(
                    folder_name + "datasets/", train=True, download=True,
                    transform=transform_test)
        indices = list(range(len(trainset)))
        np.random.RandomState(random_seed).shuffle(indices)
        indices_train = indices[:len(indices) // 2]

        indices_test = indices[len(indices) // 2:]

        train_loader = torch.utils.data.DataLoader(trainset, batch_size= batch_size, num_workers=2,
                                                sampler=torch.utils.data.sampler.SubsetRandomSampler(indices_train))

        test_loader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, num_workers=2,
                                                sampler=torch.utils.data.sampler.SubsetRandomSampler(indices_test))


        return train_loader, test_loader

    if data == 'cifar10':
        transform_test = transforms.Compose([
            transforms.ToTensor(),
        ])

        trainset = datasets.CIFAR10(
                    folder_name + "datasets/", train=True, download=True,
                    transform=transform_test)
        indices = list(range(len(trainset)))
        np.random.RandomState(random_seed).shuffle(indices)

        indices_task_train = indices[:len(indices) // 2]

        train_loader = torch.utils.data.DataLoader(trainset, batch_size= batch_size, num_workers=2,
                                                sampler=torch.utils.data.sampler.SubsetRandomSampler(indices_task_train))

        testset = datasets.MNIST(
                    folder_name + "datasets/", train=False, download=True,
                    transform=transform_test)
        test_loader = torch.utils.data.DataLoader(testset, batch_size= batch_size, num_workers=2)



        return train_loader, test_loader


#######################################
####################################### while optimizing an optimizee with trained optimizer
#######################################

def data_loader_optimizee(data = 'mnist', batch_size = 128, random_seed = 10):

    if data == 'mnist':

        transform_test = transforms.Compose([
            transforms.ToTensor(),
        ])

        trainset = datasets.MNIST(
                    folder_name + "datasets/", train=True, download=True,
                    transform=transform_test)
        indices = list(range(len(trainset)))
        np.random.RandomState(random_seed).shuffle(indices)

        indices_task_train = indices[len(indices) // 2:]

        train_loader = torch.utils.data.DataLoader(trainset, batch_size= batch_size, num_workers=2,
                                                sampler=torch.utils.data.sampler.SubsetRandomSampler(indices_task_train))

        testset = datasets.MNIST(
                    folder_name + "datasets/", train=False, download=True,
                    transform=transform_test)
        test_loader = torch.utils.data.DataLoader(testset, batch_size= batch_size, num_workers=2)


        return train_loader, test_loader

    if data == 'cifar10':
        transform_test = transforms.Compose([
            transforms.ToTensor(),
        ])

        trainset = datasets.CIFAR10(
                    folder_name + "datasets/", train=True, download=True,
                    transform=transform_test)

        train_loader = torch.utils.data.DataLoader(trainset, batch_size= batch_size, num_workers=2)

        testset = datasets.CIFAR10(
                    folder_name + "datasets/", train=False, download=True,
                    transform=transform_test)
        test_loader = torch.utils.data.DataLoader(testset, batch_size= batch_size, num_workers=2)



        return train_loader, test_loader


## Training functions

In [53]:
class Arguments:
    def __init__(self):
        # Model
        self.arch = 'resnet18_c'
        self.workers = 4
        self.epochs = 200
        self.batch_size = 128
        self.lr = 0.1
        self.weight_decay = 5e-4
        self.momentum = 0.9
        self.epochs_decay = [30, 60]
        self.print_freq = 100
        self.pretrained = False
        self.world_size = -1
        self.rank = -1
        self.dist_url = 'tcp://224.66.41.62:23456'
        self.dist_backend = 'nccl'
        self.seed = 0
        self.gpu = 0
        self.multiprocessing_distributed = False
        self.benchmark = True

        # Data
        self.cifar10_path = './data/cifar10'
        self.cifar100_path = './data/cifar100'
        self.dataset = 'CIFAR100'
        self.log_base = './results'

        # Optimization
        self.base_opt = 'SGD'
        self.grad_beta_0 = 1.
        self.grad_beta_1 = 1.
        self.grad_beta_2 = -1.
        self.grad_beta_3 = 1.
        self.grad_rho = 0.02
        self.grad_norm_rho = 0.2
        self.adaptive = False
        self.label_smoothing = 0.1
        self.grad_gamma = 0.03


        #learned optimization
        self.truncated_bptt_step = 20
        self.if_LO = True


In [64]:

import numpy as np
import torch
import torch.nn.functional as F
import torch.optim as optim
import gc
import os
# from hessian_eigenthings import compute_hessian_eigenthings
import copy





def train_opt_func(
        truncated_bptt_step,
        num_unroll_train,
        train_period,
        train_data_loader,
        dataset,
        model_name,
        device,
        meta_optimizer):
    '''
    general training function for a fixed value of
     - num_unroll_train :number of unrolling steps (inner training steps)
     - train_period : maximum epochs of training the optimizer(args.max_epochs)
     - data_loader : for training the optimizee
     - model_train : the optimizee model used for training
     - save_root : where the intermediate/best model while training, goes to

    '''


    # Create a meta optimizer that wraps a model into a meta model
    # to keep track of the meta updates.
    print(f"N_train : {num_unroll_train}, epochs = {train_period}")

    meta_model = initialize_model(dataset, model_name)


    meta_model = meta_model.to(device)


    optimizer = optim.Adam(meta_optimizer.parameters())

    #t_period many epochs with the same value of N_train
    for epoch in range(train_period):

        train_iter = iter(train_data_loader)

        print(f"optimizee initialization no. {epoch}")
        # Sample a new model
        model = initialize_model(dataset, model_name)

        model = model.to(device)

        # number of unrolls
        for k in (range(num_unroll_train // truncated_bptt_step)):


            meta_optimizer.reset_lstm(keep_states=k > 0, model=model)

            loss_sum = torch.zeros(1, requires_grad = True)
            loss_sum = loss_sum.to(device)

            # prev_loss = torch.zeros(1, requires_grad = True)
            # prev_loss = prev_loss.to(device)

            #one unrolling step
            for j in range(truncated_bptt_step):
                try:
                    x, y = next(train_iter)
                except StopIteration:
                    train_iter = iter(train_data_loader)
                    x, y = next(train_iter)
                x, y = x.to(device), y.to(device)

                # First we need to compute the gradients wrt model parameters at a point
                f_x = model(x)
                loss = F.nll_loss(f_x, y)

                model.zero_grad()
                loss.backward()

                #Update the values of the parameters using the meta optimizer, meta_model is the new model params
                meta_model = meta_optimizer.meta_update(model)

                # Compute a loss for a step the meta optimizer
                #need to find it again since cannot backward throught the same graph twice using the same loss
                # f_x = meta_model(x)
                # loss = F.nll_loss(f_x, y)
                # print(loss.item())
                # loss_sum += (j * truncated_bptt_step) * (loss - Variable(prev_loss))
                # loss_sum += loss
                # print(loss_sum.item(), loss.item())
                # prev_loss.data = loss.data

                # t = torch.cuda.get_device_properties(0).total_memory
                # r = torch.cuda.memory_reserved(0)
                # a = torch.cuda.memory_allocated(0)
                # print(f"epoch {epoch} iter {k}, {j} : {t}, {r}, {a}, {r-a}")
            #finding the loss for optimizer using the last model params
                if j == truncated_bptt_step-1:
                        print(f"epoch>> {epoch}, unrolling>>> {k}, loss on last iteration : {loss.item()} ", flush= True)

            for j in range(len(train_data_loader)):
                try:
                    x, y = next(train_iter)
                except StopIteration:
                    train_iter = iter(train_data_loader)
                    x, y = next(train_iter)
                x, y = x.to(device), y.to(device)
                f_x = meta_model(x)
                loss = F.nll_loss(f_x, y)
                loss_sum += loss

            loss_sum= loss_sum/len(train_data_loader)
            # Update the parameters of the meta optimizer
            #truncated BPTT
            optimizer.zero_grad()
            loss_sum.backward()
            optimizer.step()
            # for param in meta_optimizer.parameters():
            #     # print(param.grad)
            #     param.grad.data.clamp_(-1, 1)
            #     # print('yes')



    # del meta_model, model, x, y, prev_loss, loss, loss_sum
    gc.collect()


def reg_train_opt_func(
        truncated_bptt_step,
        num_unroll_train,
        train_period,
        train_data_loader,
        dataset,
        model_name,
        device,
        meta_optimizer,
        adaptive,
        if_smooth,
        smooth_reg_object,
        weight_decay=None):
    '''
    general training function for a fixed value of
     - num_unroll_train :number of unrolling steps (inner training steps)
     - train_period : maximum epochs of training the optimizer(args.max_epochs)
     - data_loader : for training the optimizee
     - model_train : the optimizee model used for training
     - save_root : where the intermediate/best model while training, goes to

    '''


    # Create a meta optimizer that wraps a model into a meta model
    # to keep track of the meta updates.
    print(f"N_train : {num_unroll_train}, epochs = {train_period}, ")




    base_optimizer = optim.Adam
    optimizer = SAM(meta_optimizer.parameters(), base_optimizer, adaptive=adaptive, lr=0.01)


    #t_period many epochs with the same value of N_train
    for epoch in range(train_period):

        train_iter = iter(train_data_loader)

        print(f"optimizee initialization no. {epoch}")

        # Sample a new model
        model = initialize_model(dataset, model_name)
        model = model.to(device)
        model2 = initialize_model(dataset, model_name)
        model2 = model2.to(device)

        # number of unrolls
        for k in (range(num_unroll_train // truncated_bptt_step)):

            meta_optimizer.reset_lstm(keep_states=k > 0, model=model)   #resets the initialized model parameters to that of the model

            #copying the paramters of the model to meta_model (required in SAM)
            for modelA, modelB in zip(model2.parameters(), model.parameters()):
                modelA.data.copy_(modelB.data)

            smooth_loss = torch.zeros(1, requires_grad = True)
            smooth_loss = smooth_loss.to(device)



            #one unrolling step
            for j in range(truncated_bptt_step):
                try:
                    x, y = next(train_iter)
                except StopIteration:
                    train_iter = iter(train_data_loader)
                    x, y = next(train_iter)
                x, y = x.to(device), y.to(device)

                # First we need to compute the gradients wrt model parameters at a point
                f_x = model(x)
                loss = F.nll_loss(f_x, y)
                model.zero_grad()
                loss.backward() #meta_optimizer params dont change

                #Update the values of the parameters using the meta optimizer, meta_model is the new model params
                model_temp = meta_optimizer.meta_update(model)  #model_temp points to the model used in initializing Meta_optimizer
                                                                #which will only update the meta_optimizer

                ################################
                #smoothness loss



                ################################

                if j == truncated_bptt_step-1:
                    print(f"epoch>> {epoch}, unrolling>>> {k}, loss on last iteration : {loss.item()} ", flush= True)


            #find the meta loss of the meta optimizer
            loss_sum = torch.zeros(1, requires_grad = True)
            loss_sum = loss_sum.to(device)

            for j in range(len(train_data_loader)):
                try:
                    x, y = next(train_iter)
                except StopIteration:
                    train_iter = iter(train_data_loader)
                    x, y = next(train_iter)
                x, y = x.to(device), y.to(device)
                f_x = model_temp(x)
                loss = F.nll_loss(f_x, y)
                loss_sum += loss
            loss_sum = loss_sum/len(train_data_loader)

            if if_smooth:
                smooth_loss += smooth_reg_object.run(model_temp)

            loss_sum = loss_sum + smooth_reg_object.lamb * smooth_loss


            # # Update the parameters of the meta optimizer
            # #truncated BPTT
            meta_optimizer.zero_grad()
            loss_sum.backward() #model_temp grad dont change

            #the previous step gives us the gradient on the optimizer model to be used for finding e^(w)


            #################################################################################################################
            #using the method of Sharpness Aware Minim. to obtain better params for L. Optimizer
            # meta_optimizer.reset_lstm(keep_states=k > 0, model=model)

            #1.
            ##find e^(w) and change the value of the paramerter of meta_optimizer by w + e^(w)
            optimizer.first_step(zero_grad=True)


            #2.
            #find the loss due to the w+e^(w)

            meta_optimizer.reset_lstm(keep_states=k > 0, model=model2)  #important step to reset initialized optimizee inside Meta_optimizer


            smooth_loss2 = torch.zeros(1, requires_grad = True)
            smooth_loss2 = smooth_loss2.to(device)

            #finding the optimizee parameter at the last iteration for BPTT
            for j in range(truncated_bptt_step):
                try:
                    x, y = next(train_iter)
                except StopIteration:
                    train_iter = iter(train_data_loader)
                    x, y = next(train_iter)
                x, y = x.to(device), y.to(device)
                f_x = model2(x)                 # or, model(x)
                loss = F.nll_loss(f_x, y)
                model2.zero_grad()
                loss.backward()
                model2_temp = meta_optimizer.meta_update(model2)

            #####################
            #smoothness loss
            if if_smooth:
                smooth_loss2 += smooth_reg_object.run(model2_temp)
            #####################

            wd_loss = torch.zeros(1, requires_grad = True)
            wd_loss = wd_loss.to(device)
            if weight_decay:
                wd_loss += torch.norm(get_flat_params(model2_temp).flatten(), p=2)

            loss_sum2 = torch.zeros(1, requires_grad = True)
            loss_sum2 = loss_sum2.to(device)


            #finding the optimizer's loss using the optimizee's state
            for j in range(len(train_data_loader)):
                try:
                    x, y = next(train_iter)
                except StopIteration:
                    train_iter = iter(train_data_loader)
                    x, y = next(train_iter)
                x, y = x.to(device), y.to(device)
                f_x = model2_temp(x)
                loss = F.nll_loss(f_x, y)
                loss_sum2 += loss
            loss_sum2 = loss_sum2/len(train_data_loader)
            loss_sum2 = loss_sum2 + smooth_reg_object.lamb * smooth_loss2 + weight_decay * wd_loss
            loss_sum2.backward()


            optimizer.second_step(zero_grad=False)

    ##########################################################################################################

            # for param in meta_optimizer.parameters():
            #     param.grad.data.clamp_(-1, 1)
    gc.collect()






def gam_reg_train_opt_func(
        truncated_bptt_step,
        num_unroll_train,
        train_period,
        train_data_loader,
        dataset,
        model_name,
        device,
        meta_optimizer,
        adaptive,
        if_smooth,
        smooth_reg_object,
        weight_decay=None,
        args= None):
    '''
    general training function for a fixed value of
     - num_unroll_train :number of unrolling steps (inner training steps)
     - train_period : maximum epochs of training the optimizer(args.max_epochs)
     - data_loader : for training the optimizee
     - model_train : the optimizee model used for training
     - save_root : where the intermediate/best model while training, goes to

    '''


    # Create a meta optimizer that wraps a model into a meta model
    # to keep track of the meta updates.
    print(f"N_train : {num_unroll_train}, epochs = {train_period}, ")




    base_optimizer = optim.Adam
    model2 = initialize_model(dataset, model_name)
    model2 = model2.to(device)
    args= Arguments()
    dict_ = {'model_optimizee' : model2, 'data_loder' : train_data_loader}
    gam_optimizer, base_optimizer, lr_scheduler, grad_rho_scheduler, grad_norm_rho_scheduler = get_optim_and_schedulers(meta_optimizer, optzee_model = model2, data_loader = train_data_loader, args = args)


    #t_period many epochs with the same value of N_train
    for epoch in range(train_period):

        train_iter = iter(train_data_loader)

        print(f"optimizee initialization no. {epoch}")

        # Sample a new model
        model = initialize_model(dataset, model_name)
        model = model.to(device)


        # number of unrolls
        for k in (range(num_unroll_train // truncated_bptt_step)):

            meta_optimizer.reset_lstm(keep_states=k > 0, model=model)   #resets the initialized model parameters to that of the model

            #copying the paramters of the model to meta_model (required in SAM)
            for modelA, modelB in zip(model2.parameters(), model.parameters()):
                modelA.data.copy_(modelB.data)

            smooth_loss = torch.zeros(1, requires_grad = True)
            smooth_loss = smooth_loss.to(device)



            #one unrolling step
            for j in range(truncated_bptt_step):
                try:
                    x, y = next(train_iter)
                except StopIteration:
                    train_iter = iter(train_data_loader)
                    x, y = next(train_iter)
                x, y = x.to(device), y.to(device)

                # First we need to compute the gradients wrt model parameters at a point
                f_x = model(x)
                loss = F.nll_loss(f_x, y)
                model.zero_grad()
                loss.backward() #meta_optimizer params dont change

                #Update the values of the parameters using the meta optimizer, meta_model is the new model params
                model_temp = meta_optimizer.meta_update(model)  #model_temp points to the model used in initializing Meta_optimizer
                                                                #which will only update the meta_optimizer

                ################################
                #smoothness loss



                ################################

                if j == truncated_bptt_step-1:
                    print(f"epoch>> {epoch}, unrolling>>> {k}, loss on last iteration : {loss.item()} ", flush= True)


            #find the meta loss of the meta optimizer
            loss_sum = torch.zeros(1, requires_grad = True)
            loss_sum = loss_sum.to(device)

            for j in range(len(train_data_loader)):
                try:
                    x, y = next(train_iter)
                except StopIteration:
                    train_iter = iter(train_data_loader)
                    x, y = next(train_iter)
                x, y = x.to(device), y.to(device)
                f_x = model_temp(x)
                loss = F.nll_loss(f_x, y)
                loss_sum += loss
            loss_sum = loss_sum/len(train_data_loader)

            if if_smooth:
                smooth_loss += smooth_reg_object.run(model_temp)

            loss_sum = loss_sum + smooth_reg_object.lamb * smooth_loss


            # # Update the parameters of the meta optimizer
            # #truncated BPTT
            meta_optimizer.zero_grad()
            loss_sum.backward() #model_temp grad dont change

            #the previous step gives us the gradient on the optimizer model to be used for finding e^(w)


            #################################################################################################################
            # employing gradient norm aware minimization


            loss_fn = F.nll_loss

            gam_optimizer.LO_set_closure(opzee_model = model2, iter_num = k, loss_fn = loss_fn, train_iter = train_iter, train_data_loader = train_data_loader)

            # update model parameters
            gam_optimizer.step()

    ##########################################################################################################

            # for param in meta_optimizer.parameters():
            #     param.grad.data.clamp_(-1, 1)
    gc.collect()


def eval_opt_func(
        num_unroll_eval,
        train_data_loader,
        dataset,
        model_name,
        device,
        meta_optimizer,
        eval_epochs):

    eval_loss = 0
    print("Evaluating optimizer:\n")
    for ep in range(eval_epochs):
        model = initialize_model(dataset, model_name)
        train_iter = iter(train_data_loader)
        model = model.to(device)

        loss_sum = 0
        for j in range(num_unroll_eval):
            try:
                x, y = next(train_iter)
            except StopIteration:
                train_iter = iter(train_data_loader)
                x, y = next(train_iter)
            x, y = x.to(device), y.to(device)

            # First we need to compute the gradients wrt model parameters at a point
            f_x = model(x)
            loss = F.nll_loss(f_x, y)
            model.zero_grad()
            loss.backward()
            loss_sum += loss.item()
            if j%20 == 0:
                print(f"loss: {loss.item()}\n")
            # print(loss.item()-prev_loss.item(), loss.item())

            with torch.no_grad():
                meta_optimizer.meta_update(model)
        del model, loss
        gc.collect()

        eval_loss += loss_sum/num_unroll_eval
        print(f"Evaluation Epoch {ep}| loss : {loss_sum/num_unroll_eval}")

    return eval_loss/eval_epochs





#############################################################################################

#  Hessian Eigen Value Computation

def compute_hessian_spectrum(num_eigenvals, model, train_data_loader):
    num_eigenthings = num_eigenvals  # compute top 100 eigenvalues/eigenvectors
    loss_e = nn.NLLLoss()
    eigenvals, eigenvecs = compute_hessian_eigenthings(model, train_data_loader,loss_e, num_eigenthings)
    return eigenvals.tolist(),eigenvecs.tolist()




#############################################################################################

# accuracy

def accuracy(outputs, labels):

    _, predicted = torch.max(outputs, 1)
    # print(outputs.size(), predicted.size())
    correct = (predicted == labels).sum().item()
    total = labels.size(0)
    acc = correct / total
    return acc




#############################################################################################



def compute_val_loss(test_data_loader, model):
    v_loss = 0
    test_iter = iter(test_data_loader)
    with torch.no_grad():
        for e in range(len(test_data_loader)):
            try:
                x, y = next(test_iter)
                x, y = x.to(device), y.to(device)
                f_x = model(x)
                v_loss += F.nll_loss(f_x, y)
                if e == 0:
                    pred_y = f_x
                    act_y = y
                else:
                    torch.cat([pred_y, f_x], dim = 0)
                    torch.cat([act_y, y], dim = 0)

            except StopIteration:
                test_iter = iter(test_data_loader)

    return v_loss/len(test_data_loader)





#############################################################################################
#get flat parameters

def get_flat_params(model):
    params = []
    for name, module in model.named_modules():
        if len(module._parameters) != 0:
            params.append(module._parameters['weight'].view(-1))
            try:
                params.append(module._parameters['bias'].view(-1))
            except:
                pass
    return torch.cat(params)




#set flat params

def set_flat_params(model, flat_params):
    # Restore original shapes
    offset = 0
    for module in model.modules():
        if len(module._parameters) != 0:
            weight_shape = module._parameters['weight'].size()
            weight_flat_size = reduce(mul, weight_shape, 1)
            module._parameters['weight'] = flat_params[
                                            offset:offset + weight_flat_size].view(*weight_shape)
            try:
                bias_shape = module._parameters['bias'].size()
                bias_flat_size = reduce(mul, bias_shape, 1)
                module._parameters['bias'] = flat_params[
                                                offset + weight_flat_size:offset + weight_flat_size + bias_flat_size].view(
                    *bias_shape)
            except:
                bias_flat_size = 0
            offset += weight_flat_size + bias_flat_size
    return model





def copy_gradients(from_model, to_model):
    for from_param, to_param in zip(from_model.parameters(), to_model.parameters()):
        if from_param.grad is not None:
            if to_param.grad is None:
                to_param.grad = torch.zeros_like(to_param)
                to_param.grad.data.copy_(from_param.grad.detach().data)
                # print('in function:', to_param.grad)






#############################################################################################

# Max loss in the neighbourhood
# using projected gradient ascent


def max_loss_PGD(model, eps, test_data_loader, device):

    #deep copy of the model, will be used for perturbation
    model_p = copy.deepcopy(model)
    step_size = eps/10
    #perturbation
    with torch.no_grad():
        for params in model_p.parameters():
            params.add_(torch.randn(params.size()).to(device))


    # for param_orig, param_copy in zip(model.parameters(), model_p.parameters()):
    #     assert torch.ne(param_orig.data, param_copy.data), "Parameters are not equal"

    actual_loss = compute_val_loss(test_data_loader, model)
    print(f"The actual test loss here is : {actual_loss}")

    flat_params = get_flat_params(model_p)
    perturbed_loss = []
    for _ in range(10):

        flat_params.requires_grad_()
        with torch.enable_grad():
            v_loss = 0
            test_iter = iter(test_data_loader)

            model_p = set_flat_params(model_p, flat_params)

            for e in range(len(test_data_loader)):
                try:
                    x, y = next(test_iter)
                    x, y = x.to(device), y.to(device)
                    f_x = model_p(x)
                    v_loss += F.nll_loss(f_x, y)
                    if e == 0:
                        pred_y = f_x
                        act_y = y
                    else:
                        torch.cat([pred_y, f_x], dim = 0)
                        torch.cat([act_y, y], dim = 0)

                except StopIteration:
                    test_iter = iter(test_data_loader)

            p_loss = v_loss/len(test_data_loader)
            perturbed_loss.append(p_loss.item())

            print(f"perturbed loss {_} : {perturbed_loss}")

            diff = p_loss - actual_loss

            grads = torch.autograd.grad(diff, [flat_params],allow_unused=True)[0]

            # for param_p, param, grad in zip(model_p.parameters(), model.parameters(), grads):
            flat_params = flat_params.detach() + step_size * torch.sign(grads.detach())
            flat_params = torch.min(torch.max(flat_params, flat_params - eps), flat_params + eps)

    perturbed_params = flat_params.detach()
    model_p = set_flat_params(model_p, perturbed_params)

    perturbed_loss.append(compute_val_loss(test_data_loader, model_p).detach().cpu().item())

    return perturbed_loss, actual_loss.item()





# Training LO

In [65]:
#################################################################

#           Train the optimizer model

#################################################################
import numpy as np
import tqdm
import argparse
import os
import random
import torch
import torch.nn.functional as F
import torch.optim as optim
import copy
from torch.autograd import Variable
from torchvision import datasets, transforms
from tqdm import tqdm


from datetime import datetime
import os


class OptimizationArgs:
    def __init__(self):
        self.batch_size = 128
        self.seed = 1
        self.optimizee_steps = 500
        self.truncated_bptt_step = 20
        self.max_epochs = 10000
        self.hidden_size = 20
        self.num_layers = 2
        self.save = '/content/drive/MyDrive/Mtech Project/codes/gam/GAM/gam'
        self.cuda_ = True
        self.if_cl = False
        self.dataset = 'mnist'
        self.optimizee_model = 'mlp'
        self.n_period = 3
        self.t_period = 20
        self.eval_epochs = 5
        self.if_avreg = True
        self.if_smooth = False
        self.lamb = 1.0
        self.eps = 1.0
        self.num_pga_steps = 10
        self.weight_decay = 0.005
        self.adaptive = False
        self.model_name = "unamed_model"

# Usage example:
args2 = OptimizationArgs()
args= Arguments()
print(args2)

if args2.cuda_ and torch.cuda.is_available():
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

print(f"Cuda available: {torch.cuda.is_available()} , device name : {device}")

assert args2.optimizee_steps % args2.truncated_bptt_step == 0

random.seed(args2.seed)
torch.manual_seed(args2.seed)

#directory
save_root = args2.save


#dataloaders
if (args2.dataset == 'mnist'):
    train_data_loader, test_data_loader = data_loader("mnist", batch_size = 32, random_seed = 10)

torch.cuda.empty_cache()

def main():

    loss_min = 99999.9999
    n = 1

    if args2.if_cl == True:
        N_train_steps = [100, 200, 500, 1000]
        N_eval_steps = N_train_steps[1:]
        curriculum_idx = 0
        num_unroll_train = N_train_steps[curriculum_idx]
        num_unroll_eval = N_eval_steps[curriculum_idx]

    else:
        num_unroll_train = args2.optimizee_steps
        num_unroll_eval = args2.optimizee_steps

    # # Start recording memory snapshot history
    # start_record_memory_history()


    meta_model = initialize_model(args2.dataset, args2.optimizee_model)

    meta_model = meta_model.to(device)


    meta_optimizer = MetaOptimizerLSTM(MetaModel(meta_model), args2.num_layers, args2.hidden_size, preproc_factor = 10.0, preproc =True)

    meta_optimizer = meta_optimizer.to(device)


    optimizer_path = os.path.join(save_root, f'{args2.model_name}.pt')
    counter = 0
    while os.path.exists(optimizer_path):
        optimizer_path = os.path.join(save_root, f'{args2.model_name}_{counter}.pt')
        counter += 1




    #############_________________________

    smooth_reg_object = smooth_regularize(args2.lamb, args2.eps, args2.num_pga_steps, meta_optimizer)


    improved = False
    for eph in range(args2.max_epochs//args2.t_period):

        if args2.if_avreg:
            gam_reg_train_opt_func(args2.truncated_bptt_step,
                               num_unroll_train,
                               args2.t_period,
                               train_data_loader,
                               args2.dataset,
                               args2.optimizee_model,
                               device,
                               meta_optimizer,
                               args2.adaptive,
                               args2.if_smooth,
                               smooth_reg_object,
                               args2.weight_decay,
                               args)
        else:
            train_opt_func(args2.truncated_bptt_step,
                        num_unroll_train,
                        args2.t_period,
                        train_data_loader,
                        args2.dataset,
                        args2.optimizee_model,
                        device,
                        meta_optimizer)

        torch.cuda.empty_cache()


        #evaluation of the optimizer
        n = n + 1


        loss_eval = eval_opt_func(num_unroll_eval,
                                train_data_loader,
                                args2.dataset,
                                args2.optimizee_model,
                                device,
                                meta_optimizer,
                                args2.eval_epochs)
        torch.cuda.empty_cache()

        if not args2.if_cl:
            if loss_eval < loss_min:
                loss_min == loss_eval

            #save model here
            if not os.path.exists(save_root):
                os.mkdir(save_root)
            # current_date = datetime.now().strftime("%d%m%Y")

            #saving the model
            # optimizer_path = os.path.join(save_root, f'{args.model_name}.pt')
            # counter = 0
            # while os.path.exists(optimizer_path):
            #     optimizer_path = os.path.join(save_root, f'{args.model_name}_{counter}.pt')
            #     counter += 1

            torch.save(meta_optimizer.state_dict(), optimizer_path)

            continue



        #Curriculum Learning

        if loss_eval < loss_min:

            loss_min = loss_eval
            improved = True

            #save model here
            if not os.path.exists(save_root):
                os.mkdir(save_root)
            best_meta_optimizer = copy.deepcopy(meta_optimizer)

            #saving the model
            # optimizer_path = os.path.join(save_root, f'{args.model_name}.pt')
            # counter = 0
            # while os.path.exists(optimizer_path):
            #     optimizer_path = os.path.join(save_root, f'{args.model_name}_{counter}.pt')
            #     counter += 1

            torch.save(meta_optimizer.state_dict(), optimizer_path)

        if args2.if_cl == True:
            if n >= args2.n_period and improved:

                print(f"\n\nCurriculum stage {curriculum_idx} completed !!\n\n\n")
                n = 1
                curriculum_idx += 1
                if curriculum_idx >= len(N_train_steps):
                    curriculum_idx = -1
                num_unroll_train = N_train_steps[curriculum_idx]
                num_unroll_eval = N_train_steps[curriculum_idx]


                #initial evaluation for next curriculum
                loss_min =  eval_opt_func(num_unroll_eval,
                                        train_data_loader,
                                        args2.dataset,
                                        args2.optimizee_model,
                                        device,
                                        best_meta_optimizer,
                                        args2.eval_epochs)
                torch.cuda.empty_cache()


            elif n >= args2.n_period and not improved:
                print(f"No improvement with curriculum {curriculum_idx} : STOPPING Here!")
                break




Cuda available: True , device name : cuda:0


In [66]:
main()

N_train : 500, epochs = 20, 
optimizee initialization no. 0
epoch>> 0, unrolling>>> 0, loss on last iteration : 2.112440586090088 


UnboundLocalError: local variable 'train_iter' referenced before assignment

# GAM test

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Hyperparameters
input_size = 28 * 28  # MNIST image size is 28x28
hidden_size = 20
num_classes = 10
num_epochs = 10
batch_size = 128
learning_rate = 0.001

# MNIST dataset
train_dataset = torchvision.datasets.MNIST(root='./data',
                                           train=True,
                                           transform=transforms.ToTensor(),
                                           download=True)

test_dataset = torchvision.datasets.MNIST(root='./data',
                                          train=False,
                                          transform=transforms.ToTensor())

# Data loader
train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                           batch_size=batch_size,
                                           shuffle=True)

test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                          batch_size=batch_size,
                                          shuffle=False)

# Fully connected neural network with one hidden layer
class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

model = MLP(input_size, hidden_size, num_classes).to(device)

# # Loss and optimizer
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# # Train the model
# total_step = len(train_loader)
# for epoch in range(num_epochs):
#     for i, (images, labels) in enumerate(train_loader):
#         # Move tensors to the configured device
#         images = images.reshape(-1, 28*28).to(device)
#         labels = labels.to(device)

#         # Forward pass
#         outputs = model(images)
#         loss = criterion(outputs, labels)

#         # Backward and optimize
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()

#         if (i+1) % 100 == 0:
#             print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}'
#                    .format(epoch+1, num_epochs, i+1, total_step, loss.item()))



In [8]:
# for x, y in train_loader:
#     print(x.squeeze().shape, y.shape)

In [9]:
class Arguments:
    def __init__(self):
        # Model
        self.arch = 'resnet18_c'
        self.workers = 4
        self.epochs = 200
        self.batch_size = 128
        self.lr = 0.1
        self.weight_decay = 5e-4
        self.momentum = 0.9
        self.epochs_decay = [30, 60]
        self.print_freq = 100
        self.pretrained = False
        self.world_size = -1
        self.rank = -1
        self.dist_url = 'tcp://224.66.41.62:23456'
        self.dist_backend = 'nccl'
        self.seed = 0
        self.gpu = 0
        self.multiprocessing_distributed = False
        self.benchmark = True

        # Data
        self.cifar10_path = './data/cifar10'
        self.cifar100_path = './data/cifar100'
        self.dataset = 'CIFAR100'
        self.log_path = './results'

        # Optimization
        self.base_opt = 'SGD'
        self.grad_beta_0 = 1.
        self.grad_beta_1 = 1.
        self.grad_beta_2 = -1.
        self.grad_beta_3 = 1.
        self.grad_rho = 0.02
        self.grad_norm_rho = 0.2
        self.adaptive = False
        self.label_smoothing = 0.1
        self.grad_gamma = 0.03




In [10]:
lr = 0.01
momentum = 0
weight_decay = 0
args = Arguments()

In [22]:
inputs.size()[0]

128

In [24]:
# optimizer initialization
# initialize a base optimizer, such as SGD, Adam, AdamW ...
base_optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)

# initialize GAM optimizer based on the base optimizer
# gam_optimizer = gam.GAM(params=model.parameters(), base_optimizer=base_optimizer, model=model)
# gam_optimizer.update_rho_t()
# print(gam_optimizer.grad_rho_scheduler)
# training
# define the loss function for loss and gradients
def loss_fn(predictions, targets):
    return sce.smooth_crossentropy(predictions, targets, smoothing=0.1).mean()

gam_optimizer, base_optimizer, lr_scheduler, grad_rho_scheduler, grad_norm_rho_scheduler = get_optim_and_schedulers(model, args)

for inputs, labels in train_loader:
    # get inputs and labels
    print(inputs.shape)
    inputs = inputs.squeeze().view(inputs.size()[0],-1).cuda()
    # print(inputs.shape)
    labels = labels.cuda()

    # GAM sets closure and automatically runs predictions = model(inputs), loss = loss_fn(predictions, targets), loss.backward() in it
    gam_optimizer.set_closure(loss_fn, inputs, labels)

    # update model parameters
    predictions, loss = gam_optimizer.step()


torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([128, 1, 28, 28])
torch.Size([12

In [12]:
class Arguments:
    def __init__(self):
        # Model
        self.arch = 'resnet18_c'
        self.workers = 4
        self.epochs = 200
        self.batch_size = 128
        self.lr = 0.1
        self.weight_decay = 5e-4
        self.momentum = 0.9
        self.epochs_decay = [30, 60]
        self.print_freq = 100
        self.pretrained = False
        self.world_size = -1
        self.rank = -1
        self.dist_url = 'tcp://224.66.41.62:23456'
        self.dist_backend = 'nccl'
        self.seed = 0
        self.gpu = 0
        self.multiprocessing_distributed = False
        self.benchmark = True

        # Data
        self.cifar10_path = './data/cifar10'
        self.cifar100_path = './data/cifar100'
        self.dataset = 'CIFAR100'
        self.log_base = './results'

        # Optimization
        self.base_opt = 'SGD'
        self.grad_beta_0 = 1.
        self.grad_beta_1 = 1.
        self.grad_beta_2 = -1.
        self.grad_beta_3 = 1.
        self.grad_rho = 0.02
        self.grad_norm_rho = 0.2
        self.adaptive = False
        self.label_smoothing = 0.1
        self.grad_gamma = 0.03


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/Mtech Project/codes/gam/GAM')

In [13]:
import argparse
import os
import random
import warnings
import models
import numpy as np

import torch
import torch.nn as nn
import torch.nn.parallel
import torch.backends.cudnn as cudnn
import torch.distributed as dist
import torch.optim
import torch.multiprocessing as mp
import torch.utils.data
import torch.utils.data.distributed
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.tensorboard import SummaryWriter

from gam.validate import validate
from gam.train_with_gam import train_epoch
from gam.optimizer_helper import get_optim_and_schedulers

model_names = sorted(name for name in models.__dict__
                     if name.islower() and not name.startswith("__")
                     and callable(models.__dict__[name]))

print('model name space', model_names)

parser = argparse.ArgumentParser(description='PyTorch CIFAR Training with GAM')



# Create an instance of the Arguments class
args = Arguments()

return_acc = 0

model_names = sorted(name for name in models.__dict__
                     if name.islower() and not name.startswith("__")
                     and callable(models.__dict__[name]))


def main():
    torch.backends.cudnn.benchmark = True
    #CuDNN offers multiple algorithms for convolution operations, including direct convolution,
    #FFT-based convolution, and Winograd-based convolution. These algorithms have different
    #performance characteristics depending on factors like input size, filter size, and hardware architecture.
    #By enabling benchmarking (torch.backends.cudnn.benchmark = True), CuDNN measures the
    #performance of these algorithms on the current hardware and dynamically selects the fastest
    #algorithm for each operation during the first forward pass of your model.

    args = parser.parse_args()

    # default hps
    if args.dataset == 'CIFAR100':
        if args.arch.startswith('resnet'):
            args.grad_norm_rho, args.grad_rho, args.grad_beta_0, args.grad_beta_1, args.grad_gamma = 0.2, 0.02, 0.5, 0.6, 0.03
        elif args.arch.startswith('pyramidnet'):
            args.grad_norm_rho, args.grad_rho, args.grad_beta_0, args.grad_beta_1, args.grad_gamma = 0.2, 0.04, 0.3, 0.5, 0.05
    elif args.dataset == 'CIFAR10':
        if args.arch.startswith('resnet'):
            args.grad_norm_rho, args.grad_rho, args.grad_beta_0, args.grad_beta_1, args.grad_gamma = 0.2, 0.03, 0.1, 0.1, 0.05
        elif args.arch.startswith('pyramidnet'):
            args.grad_norm_rho, args.grad_rho, args.grad_beta_0, args.grad_beta_1, args.grad_gamma = 0.2, 0.03, 0.1, 0.1, 0.03

    args.grad_beta_2 = 1 - args.grad_beta_0
    args.grad_beta_3 = 1 - args.grad_beta_1

    log_description = 'GAM'
    args.log_path = os.path.join(args.log_base, args.dataset, log_description, "log.txt")

    if args.seed is not None:
        # for reimplement
        random.seed(args.seed)
        torch.manual_seed(args.seed)
        os.environ['PYTHONHASHSEED'] = str(args.seed)
        torch.cuda.manual_seed(args.seed)
        torch.cuda.manual_seed_all(args.seed)
        np.random.seed(args.seed)

        torch.backends.cudnn.deterministic = True
        #By setting torch.backends.cudnn.deterministic = True, you ensure that the CuDNN library
        #always uses deterministic algorithms for its operations. This means that, given the
        #same input data and the same set of parameters, the results of CuDNN operations will
        #always be the same, regardless of the underlying hardware or other environmental factors.


        warnings.warn('You have chosen to seed training. '
                      'This will turn on the CUDNN deterministic setting, '
                      'which can slow down your training considerably! '
                      'You may see unexpected behavior when restarting '
                      'from checkpoints.')

    if args.gpu is not None:
        warnings.warn('You have chosen a specific GPU. This will completely '
                      'disable data parallelism.')

    if args.dist_url == "env://" and args.world_size == -1:
        args.world_size = int(os.environ["WORLD_SIZE"])

    args.distributed = args.world_size > 1 or args.multiprocessing_distributed

    ngpus_per_node = torch.cuda.device_count()
    if args.multiprocessing_distributed:
        args.world_size = ngpus_per_node * args.world_size
        mp.spawn(main_worker, nprocs=ngpus_per_node, args=(ngpus_per_node, args))
    else:
        main_worker(args.gpu, ngpus_per_node, args)


def main_worker(gpu, ngpus_per_node, args):
    global return_acc
    args.gpu = gpu

    if args.gpu is not None:
        print("Use GPU: {} for training".format(args.gpu))

    if args.distributed:
        if args.dist_url == "env://" and args.rank == -1:
            args.rank = int(os.environ["RANK"])
        if args.multiprocessing_distributed:
            args.rank = args.rank * ngpus_per_node + gpu
        dist.init_process_group(backend=args.dist_backend, init_method=args.dist_url,
                                world_size=args.world_size, rank=args.rank)
    # create model
    num_classes = 10 if args.dataset == 'CIFAR10' else 100
    if args.pretrained:
        print("=> using pre-trained model '{}'".format(args.arch))
        model = models.__dict__[args.arch](pretrained=True, num_classes=num_classes)
    else:
        print("=> creating model '{}'".format(args.arch))
        model = models.__dict__[args.arch](num_classes=num_classes)

    if args.distributed:
        if args.gpu is not None:
            torch.cuda.set_device(args.gpu)
            model.cuda(args.gpu)
            args.batch_size = int(args.batch_size / ngpus_per_node)
            args.workers = int((args.workers + ngpus_per_node - 1) / ngpus_per_node)
            model = torch.nn.parallel.DistributedDataParallel(model, device_ids=[args.gpu])
        else:
            model.cuda()
            model = torch.nn.parallel.DistributedDataParallel(model)
    elif args.gpu is not None:
        torch.cuda.set_device(args.gpu)
        model = model.cuda(args.gpu)
    else:
        model = torch.nn.DataParallel(model).cuda()

    # define loss function (criterion) and optimizer
    criterion = nn.CrossEntropyLoss().cuda(args.gpu)
    cudnn.benchmark = args.benchmark

    if args.dataset == 'CIFAR10':
        data_root = args.cifar10_path
        train_dataset = datasets.CIFAR10(
            root=data_root,
            train=True,
            download=True,
            transform=transforms.Compose([
                transforms.RandomCrop(32, padding=4),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.49139968, 0.48215827, 0.44653124], std=[0.2023, 0.1994, 0.2010])
            ]))

        test_dataset = datasets.CIFAR10(
            root=data_root,
            train=False,
            download=True,
            transform=transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.49139968, 0.48215827, 0.44653124], std=[0.2023, 0.1994, 0.2010])
            ]))

    else:
        data_root = args.cifar100_path
        train_dataset = datasets.CIFAR100(
            root=data_root,
            train=True,
            download=True,
            transform=transforms.Compose([
                transforms.RandomCrop(32, padding=4),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.5071, 0.4867, 0.4408], std=[0.2675, 0.2565, 0.2761])
            ]))

        test_dataset = datasets.CIFAR100(
            root=data_root,
            train=False,
            download=True,
            transform=transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.5071, 0.4867, 0.4408], std=[0.2675, 0.2565, 0.2761])
            ]))

    if args.distributed:
        train_sampler = torch.utils.data.distributed.DistributedSampler(train_dataset)
    else:
        train_sampler = None

    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=args.batch_size, shuffle=(train_sampler is None),
        num_workers=args.workers, pin_memory=True, sampler=train_sampler)

    val_loader = torch.utils.data.DataLoader(
        test_dataset,
        batch_size=args.batch_size, shuffle=False,
        num_workers=args.workers, pin_memory=True)

    log_dir = os.path.dirname(args.log_path)
    print('tensorboard dir {}'.format(log_dir))
    tensor_writer = SummaryWriter(log_dir)

    # get base opt and schedulers
    optimizer, base_optimizer, lr_scheduler, grad_rho_scheduler, grad_norm_rho_scheduler = get_optim_and_schedulers(
        model, args)

    for epoch in range(args.epochs):

        if args.distributed:
            train_sampler.set_epoch(epoch)

        train_epoch(gpu, train_loader, model, base_optimizer, epoch, args,
                    lr_scheduler=lr_scheduler, grad_rho_scheduler=grad_rho_scheduler,
                    grad_norm_rho_scheduler=grad_norm_rho_scheduler, optimizer=optimizer)

        lr_scheduler.step()

        acc1 = validate(gpu, val_loader, model, criterion, True, args)
        return_acc = max(return_acc, acc1)
        tensor_writer.add_scalar('return_ACC@1/test', return_acc, epoch)

    print('Test top-1 acc: ', return_acc)

main()


ModuleNotFoundError: No module named 'models'